In [35]:
# 1-1
import numpy as np
import pandas as pd
df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/9_1_1.csv')
# print(df.info())
# print(df.head())
df['sum']=df['금액1']+df['금액2']
# print(df.info())
# print(df.head())

df1=df.groupby(['year', '지역코드', 'gender'])['sum'].sum().reset_index()
df2=df1.pivot(index=['year', '지역코드'], columns='gender', values='sum')
df2=df2.fillna(0)
df3=np.abs(df2[0] - df2[1])
df3.sort_values(ascending=False).index[0][1]

np.int64(143)

In [38]:
# 1-2
import numpy as np
import pandas as pd
df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/9_1_2.csv')
# print(df.info())
# print(df.head())

# df['sum']=df[['범죄유형1', '범죄유형2', '범죄유형3', '범죄유형4', '범죄유형5', '범죄유형6', '범죄유형7', '범죄유형8', '범죄유형9']].sum(axis=1)
# print(df.info())
# print(df.head())

df1=df[df['구분']=='검거건수'].set_index('연도').drop(columns=['구분'])
df2=df[df['구분']=='발생건수'].set_index('연도').drop(columns=['구분'])
df3=df1/df2
def check(x):
    return x==x.max()
df4=df3.apply(check)
print(df1[df4].fillna(0).sum().sum())

36977.0


In [15]:
# 1-3
import numpy as np
import pandas as pd
df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/9_1_3.csv')
# print(df.info())
# print(df.head())
df['평균만족도']=df['평균만족도'].fillna(df['평균만족도'].mean())
df['근속연수']=df['근속연수'].fillna(np.floor(df.groupby(['부서', '등급'])['근속연수'].transform('mean')))
# print(df.info())
# print(df.head())
A=df[(df['부서']=='HR') & (df['등급']=='A')]['근속연수'].mean()
B=df[(df['부서']=='Sales') & (df['등급']=='B')]['교육참가횟수'].mean()
print(np.round((A+B),3))

25.225


In [100]:
# 2
import numpy as np
import pandas as pd
train_df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/9_2_train.csv')
test_df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/9_2_test.csv')
# print(train_df.info())
# print(train_df.head())
# print(test_df.info())

train_y=train_df['라벨']
train_X=train_df.drop(columns=['ID', '라벨'])
test_y=test_df['라벨']
test_X=test_df.drop(columns=['ID', '라벨'])
# print(train_X.info())
# print(train_X.head())
# print(train_y.info())
# print(test_X.info())
# print(test_y.info())

from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import f1_score, get_scorer_names
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

train_X, valid_X, train_y, valid_y=train_test_split(train_X, train_y, test_size=0.3)
# print(train_X.shape, train_y.shape, valid_X.shape, valid_y.shape)

obj=train_X.select_dtypes('object').columns.tolist()
num=train_X.select_dtypes('number').columns.tolist()
# print(obj,num)
encoder=OneHotEncoder(sparse_output=False, handle_unknown='ignore')
train_X_encoder=encoder.fit_transform(train_X[obj])
valid_X_encoder=encoder.transform(valid_X[obj])
test_X_encoder=encoder.transform(test_X[obj])

train_X=np.concatenate([train_X_encoder, train_X[num]],axis=1)
valid_X=np.concatenate([valid_X_encoder, valid_X[num]],axis=1)
test_X=np.concatenate([test_X_encoder, test_X[num]],axis=1)

rf=RandomForestClassifier()
# rf.fit(train_X, train_y)
# pred1=rf.predict(valid_X)
# pred1
# print(f1_score(valid_y, pred1, average='macro'))

params={
    'max_depth': [10,20,30],
    'min_samples_split': [2,5,10]
}
cv=GridSearchCV(estimator=rf, param_grid=params, scoring='f1_macro', cv=3)
# cv.fit(train_X, train_y)
# pred2=cv.predict(valid_X)
# print(f1_score(valid_y, pred2, average='macro'))

train_X_full=np.concatenate([train_X,valid_X],axis=0)
train_y_full=np.concatenate([train_y,valid_y],axis=0)

rf.fit(train_X_full, train_y_full)
pred3=rf.predict(test_X)
print(f1_score(test_y, pred3, average='macro'))

cv.fit(train_X_full, train_y_full)
pred4=cv.predict(test_X)
print(f1_score(test_y, pred4, average='macro'))

pd.DataFrame(pred4, columns=['pred']).to_csv('result.csv', index=False)

0.3073073073073073
0.3280480971635395


In [61]:
# 3-1
import numpy as np
import pandas as pd
df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/9_3_1.csv')
# print(df.info())
# print(df.head())

train_df=df.loc[:139]
test_df=df.loc[140:]
# print(train_df, test_df)
train_df=train_df.drop(columns=['id'])
train_y=train_df['design']
train_X=train_df.drop(columns=['design'])
test_y=test_df['design']
test_X=test_df.drop(columns=['design', 'id'])
# print(train_X.info())
# print(train_y.info())

import statsmodels.api as sm
import statsmodels.formula.api as smf
model=smf.ols('design ~ tenure + f2 + f3 + f4 + f5', data=train_df).fit()
# print(model.summary())
print((model.pvalues[1:] > 0.05).sum())
pred1=model.predict(train_X)
from scipy.stats import pearsonr
print(np.round(pearsonr(train_y, pred1).statistic,2))
pred2=model.predict(test_X)
from sklearn.metrics import root_mean_squared_error
print(np.round(root_mean_squared_error(test_y, pred2),0))

2
0.91
4.0


In [76]:
# 3-2
import numpy as np
import pandas as pd
df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/9_3_2.csv')
# print(df.info())
# print(df.head())
import statsmodels.api as sm
import statsmodels.formula.api as smf
model=smf.glm('churn ~ col1 + col2 + Phone_Service + Tech_Insurance', data=df, family=sm.families.Binomial()).fit()
# print(model.summary())
print(np.round(model.pvalues['col1'],3))
print(np.round(np.exp(model.params['Phone_Service']),3))
# print(dir(model))
train_X=df.drop(columns=['churn'])
pred1=model.predict(train_X)
print((pred1 >= 0.3).sum())

0.0
1.867
450
